1. READ RAW DATA FROM ADLS

In [0]:
spark.conf.set(
  "fs.azure.account.key.stdeprojectksc.dfs.core.windows.net",
  dbutils.secrets.get(scope="adls-scope", key="adls-key")
)


In [0]:
import requests
import pandas as pd
from datetime import datetime
# CoinGecko API endpoint
url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1,
    "sparkline": False
}
response = requests.get(url, params=params)
raw_json = response.json()
pdf_raw = pd.DataFrame(raw_json)
raw_df = spark.createDataFrame(pdf_raw)
# RAW path (JSON)
raw_write_path = "abfss://raw@stdeprojectksc.dfs.core.windows.net/coingecko/crypto_raw"
raw_df.write.mode("overwrite").json(raw_write_path)
print("Raw CoinGecko data written to ADLS (JSON)")


✅ Raw CoinGecko data written to ADLS (JSON)


In [0]:
raw_path = "abfss://raw@stdeprojectksc.dfs.core.windows.net/coingecko/crypto_raw.json"

df_raw = spark.read.json(raw_path)
df_raw.display()

ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,circulating_supply,current_price,fully_diluted_valuation,high_24h,id,image,last_updated,low_24h,market_cap,market_cap_change_24h,market_cap_change_percentage_24h,market_cap_rank,max_supply,name,price_change_24h,price_change_percentage_24h,roi,symbol,total_supply,total_volume
126080.0,-28.46798,2025-10-06T18:57:42.558Z,67.81,132902.23654,2013-07-06T00:00:00.000Z,1.9981509E7,90187.0,1802957654463,90273.0,bitcoin,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,2026-01-28T13:56:32.860Z,87315.0,1802957654463,48121935430,2.74225,1,2.1E7,Bitcoin,2313.08,2.63226,null,btc,1.9981509E7,49207481311
4946.05,-38.66904,2025-08-24T19:21:03.333Z,0.432979,700502.11459,2015-10-20T00:00:00.000Z,1.206943737963051E8,3033.46,366312531113,3036.85,ethereum,https://coin-images.coingecko.com/coins/images/279/large/ethereum.png?1696501628,2026-01-28T13:56:32.866Z,2908.29,366312531113,15497546212,4.41758,2,null,Ethereum,125.16,4.30368,"List(btc, 4395.8274145196565, 43.95827414519657)",eth,1.206943737963051E8,28853865931
3.65,-47.03837,2025-07-18T03:40:53.808Z,0.00268621,71792.49696,2014-05-22T00:00:00.000Z,6.0853233336E10,1.93,193095425003,1.94,ripple,https://coin-images.coingecko.com/coins/images/44/large/xrp-symbol-white-128.png?1696501442,2026-01-28T13:56:30.246Z,1.87,117521586484,3399561708,2.97888,5,1.0E11,XRP,0.056154,2.99482,null,xrp,9.9985724371E10,2432407258
293.31,-56.58574,2025-01-19T11:15:27.957Z,0.500801,25327.09865,2020-05-11T19:35:23.449Z,5.660685576819917E8,127.34,78854170037,127.87,solana,https://coin-images.coingecko.com/coins/images/4128/large/solana.png?1718769756,2026-01-28T13:56:29.408Z,123.05,72090574319,2356010258,3.37854,6,null,Solana,4.18,3.39395,null,sol,6.191775654644122E8,4343364070


2. CLEAN & STANDARDIZE DATA (RAW TO PROCESSED)
This includes:
- Select only required columns
- Rename columns
- Fix data types

In [0]:
df_raw.printSchema()

root
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: double (nullable = true)
 |-- ath_date: string (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change_percentage: double (nullable = true)
 |-- atl_date: string (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- current_price: double (nullable = true)
 |-- fully_diluted_valuation: long (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- id: string (nullable = true)
 |-- image: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_change_24h: long (nullable = true)
 |-- market_cap_change_percentage_24h: double (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- name: string (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = true)
 |-- roi: st

In [0]:
from pyspark.sql.functions import col

df_selected = df_raw.select(
    col("id").alias("coin_id"),
    col("name"),
    col("symbol"),
    col("current_price"),
    col("market_cap"),
    col("market_cap_rank"),
    col("total_volume"),
    col("price_change_percentage_24h"),
    col("last_updated")
)

df_selected.display()


coin_id,name,symbol,current_price,market_cap,market_cap_rank,total_volume,price_change_percentage_24h,last_updated
bitcoin,Bitcoin,btc,90187.0,1802957654463,1,49207481311,2.63226,2026-01-28T13:56:32.860Z
ethereum,Ethereum,eth,3033.46,366312531113,2,28853865931,4.30368,2026-01-28T13:56:32.866Z
ripple,XRP,xrp,1.93,117521586484,5,2432407258,2.99482,2026-01-28T13:56:30.246Z
solana,Solana,sol,127.34,72090574319,6,4343364070,3.39395,2026-01-28T13:56:29.408Z


In [0]:
df_cleaned = df_selected.dropna(
    subset=["current_price", "market_cap"]
)

df_cleaned.display()

coin_id,name,symbol,current_price,market_cap,market_cap_rank,total_volume,price_change_percentage_24h,last_updated
bitcoin,Bitcoin,btc,90187.0,1802957654463,1,49207481311,2.63226,2026-01-28T13:56:32.860Z
ethereum,Ethereum,eth,3033.46,366312531113,2,28853865931,4.30368,2026-01-28T13:56:32.866Z
ripple,XRP,xrp,1.93,117521586484,5,2432407258,2.99482,2026-01-28T13:56:30.246Z
solana,Solana,sol,127.34,72090574319,6,4343364070,3.39395,2026-01-28T13:56:29.408Z


In [0]:
from pyspark.sql.functions import to_timestamp, current_timestamp, lit

df_enriched = (
    df_cleaned
    .withColumn(
        "last_updated_ts",
        to_timestamp("last_updated")
    )
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("data_source", lit("coingecko_api"))
)

df_enriched.display()

coin_id,name,symbol,current_price,market_cap,market_cap_rank,total_volume,price_change_percentage_24h,last_updated,last_updated_ts,ingestion_timestamp,data_source
bitcoin,Bitcoin,btc,90187.0,1802957654463,1,49207481311,2.63226,2026-01-28T13:56:32.860Z,2026-01-28T13:56:32.86Z,2026-02-03T14:56:52.952526Z,coingecko_api
ethereum,Ethereum,eth,3033.46,366312531113,2,28853865931,4.30368,2026-01-28T13:56:32.866Z,2026-01-28T13:56:32.866Z,2026-02-03T14:56:52.952526Z,coingecko_api
ripple,XRP,xrp,1.93,117521586484,5,2432407258,2.99482,2026-01-28T13:56:30.246Z,2026-01-28T13:56:30.246Z,2026-02-03T14:56:52.952526Z,coingecko_api
solana,Solana,sol,127.34,72090574319,6,4343364070,3.39395,2026-01-28T13:56:29.408Z,2026-01-28T13:56:29.408Z,2026-02-03T14:56:52.952526Z,coingecko_api


In [0]:
processed_path = "abfss://processed@stdeprojectksc.dfs.core.windows.net/coingecko/crypto_prices"

df_enriched.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(processed_path)

In [0]:
df_processed = spark.read.parquet(processed_path)
df_processed.printSchema()
df_processed.display()

root
 |-- coin_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- total_volume: long (nullable = true)
 |-- price_change_percentage_24h: double (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- last_updated_ts: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- data_source: string (nullable = true)



coin_id,name,symbol,current_price,market_cap,market_cap_rank,total_volume,price_change_percentage_24h,last_updated,last_updated_ts,ingestion_timestamp,data_source
bitcoin,Bitcoin,btc,90187.0,1802957654463,1,49207481311,2.63226,2026-01-28T13:56:32.860Z,2026-01-28T13:56:32.86Z,2026-02-03T14:56:53.558075Z,coingecko_api
ethereum,Ethereum,eth,3033.46,366312531113,2,28853865931,4.30368,2026-01-28T13:56:32.866Z,2026-01-28T13:56:32.866Z,2026-02-03T14:56:53.558075Z,coingecko_api
ripple,XRP,xrp,1.93,117521586484,5,2432407258,2.99482,2026-01-28T13:56:30.246Z,2026-01-28T13:56:30.246Z,2026-02-03T14:56:53.558075Z,coingecko_api
solana,Solana,sol,127.34,72090574319,6,4343364070,3.39395,2026-01-28T13:56:29.408Z,2026-01-28T13:56:29.408Z,2026-02-03T14:56:53.558075Z,coingecko_api


3. CURATED LAYER (PROCESSED TO CURATED) FOR ANALYTICS
The curated layer contains: 
- Aggregated
- Business focussed
- Analytics ready datasets

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("coin_id").orderBy(col("ingestion_timestamp").desc())

df_latest = (
    df_processed
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

df_latest.display()

coin_id,name,symbol,current_price,market_cap,market_cap_rank,total_volume,price_change_percentage_24h,last_updated,last_updated_ts,ingestion_timestamp,data_source
bitcoin,Bitcoin,btc,90187.0,1802957654463,1,49207481311,2.63226,2026-01-28T13:56:32.860Z,2026-01-28T13:56:32.86Z,2026-02-03T14:56:53.558075Z,coingecko_api
ethereum,Ethereum,eth,3033.46,366312531113,2,28853865931,4.30368,2026-01-28T13:56:32.866Z,2026-01-28T13:56:32.866Z,2026-02-03T14:56:53.558075Z,coingecko_api
ripple,XRP,xrp,1.93,117521586484,5,2432407258,2.99482,2026-01-28T13:56:30.246Z,2026-01-28T13:56:30.246Z,2026-02-03T14:56:53.558075Z,coingecko_api
solana,Solana,sol,127.34,72090574319,6,4343364070,3.39395,2026-01-28T13:56:29.408Z,2026-01-28T13:56:29.408Z,2026-02-03T14:56:53.558075Z,coingecko_api


In [0]:
curated_snapshot_path = "abfss://curated@stdeprojectksc.dfs.core.windows.net/coingecko/crypto_latest_snapshot"

df_latest.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(curated_snapshot_path)

In [0]:
from pyspark.sql.functions import sum, count, max

df_market_summary = df_latest.agg(
    sum("market_cap").alias("total_market_cap"),
    sum("total_volume").alias("total_24h_volume"),
    count("coin_id").alias("number_of_coins"),
    max("ingestion_timestamp").alias("snapshot_timestamp")
)

df_market_summary.display()

total_market_cap,total_24h_volume,number_of_coins,snapshot_timestamp
2358882346379,84837118570,4,2026-02-03T14:56:53.558075Z


In [0]:
curated_summary_path = "abfss://curated@stdeprojectksc.dfs.core.windows.net/coingecko/market_summary"

df_market_summary.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(curated_summary_path)

In [0]:
spark.read.parquet(curated_snapshot_path).display()
spark.read.parquet(curated_summary_path).display()

coin_id,name,symbol,current_price,market_cap,market_cap_rank,total_volume,price_change_percentage_24h,last_updated,last_updated_ts,ingestion_timestamp,data_source
bitcoin,Bitcoin,btc,90187.0,1802957654463,1,49207481311,2.63226,2026-01-28T13:56:32.860Z,2026-01-28T13:56:32.86Z,2026-02-03T14:56:53.558075Z,coingecko_api
ethereum,Ethereum,eth,3033.46,366312531113,2,28853865931,4.30368,2026-01-28T13:56:32.866Z,2026-01-28T13:56:32.866Z,2026-02-03T14:56:53.558075Z,coingecko_api
ripple,XRP,xrp,1.93,117521586484,5,2432407258,2.99482,2026-01-28T13:56:30.246Z,2026-01-28T13:56:30.246Z,2026-02-03T14:56:53.558075Z,coingecko_api
solana,Solana,sol,127.34,72090574319,6,4343364070,3.39395,2026-01-28T13:56:29.408Z,2026-01-28T13:56:29.408Z,2026-02-03T14:56:53.558075Z,coingecko_api


total_market_cap,total_24h_volume,number_of_coins,snapshot_timestamp
2358882346379,84837118570,4,2026-02-03T14:56:53.558075Z


4. Databricks SQL Warehouse for Power BI Visualization

In [0]:
curated_base_path = "abfss://curated@stdeprojectksc.dfs.core.windows.net/crypto_analytics"

In [0]:
dbutils.fs.ls("abfss://curated@stdeprojectksc.dfs.core.windows.net/")

[FileInfo(path='abfss://curated@stdeprojectksc.dfs.core.windows.net/coingecko/', name='coingecko/', size=0, modificationTime=1769693956000),
 FileInfo(path='abfss://curated@stdeprojectksc.dfs.core.windows.net/crypto_analytics/', name='crypto_analytics/', size=0, modificationTime=1769697070000)]

In [0]:
from pyspark.sql.functions import to_timestamp

df_latest = (
    df_raw
    .withColumn("last_updated_ts", to_timestamp("last_updated"))
    .dropDuplicates(["id"])
)

In [0]:
df_latest.display()


ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,circulating_supply,current_price,fully_diluted_valuation,high_24h,id,image,last_updated,low_24h,market_cap,market_cap_change_24h,market_cap_change_percentage_24h,market_cap_rank,max_supply,name,price_change_24h,price_change_percentage_24h,roi,symbol,total_supply,total_volume,last_updated_ts
126080.0,-28.46798,2025-10-06T18:57:42.558Z,67.81,132902.23654,2013-07-06T00:00:00.000Z,1.9981509E7,90187.0,1802957654463,90273.0,bitcoin,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,2026-01-28T13:56:32.860Z,87315.0,1802957654463,48121935430,2.74225,1,2.1E7,Bitcoin,2313.08,2.63226,null,btc,1.9981509E7,49207481311,2026-01-28T13:56:32.86Z
4946.05,-38.66904,2025-08-24T19:21:03.333Z,0.432979,700502.11459,2015-10-20T00:00:00.000Z,1.206943737963051E8,3033.46,366312531113,3036.85,ethereum,https://coin-images.coingecko.com/coins/images/279/large/ethereum.png?1696501628,2026-01-28T13:56:32.866Z,2908.29,366312531113,15497546212,4.41758,2,null,Ethereum,125.16,4.30368,"List(btc, 4395.8274145196565, 43.95827414519657)",eth,1.206943737963051E8,28853865931,2026-01-28T13:56:32.866Z
3.65,-47.03837,2025-07-18T03:40:53.808Z,0.00268621,71792.49696,2014-05-22T00:00:00.000Z,6.0853233336E10,1.93,193095425003,1.94,ripple,https://coin-images.coingecko.com/coins/images/44/large/xrp-symbol-white-128.png?1696501442,2026-01-28T13:56:30.246Z,1.87,117521586484,3399561708,2.97888,5,1.0E11,XRP,0.056154,2.99482,null,xrp,9.9985724371E10,2432407258,2026-01-28T13:56:30.246Z
293.31,-56.58574,2025-01-19T11:15:27.957Z,0.500801,25327.09865,2020-05-11T19:35:23.449Z,5.660685576819917E8,127.34,78854170037,127.87,solana,https://coin-images.coingecko.com/coins/images/4128/large/solana.png?1718769756,2026-01-28T13:56:29.408Z,123.05,72090574319,2356010258,3.37854,6,null,Solana,4.18,3.39395,null,sol,6.191775654644122E8,4343364070,2026-01-28T13:56:29.408Z


In [0]:
df_market = df_latest.select(
    "id",
    "symbol",
    "name",
    "current_price",
    "market_cap",
    "market_cap_rank",
    "total_volume",
    "circulating_supply",
    "last_updated_ts"
)


In [0]:
df_market.display()

id,symbol,name,current_price,market_cap,market_cap_rank,total_volume,circulating_supply,last_updated_ts
bitcoin,btc,Bitcoin,90187.0,1802957654463,1,49207481311,1.9981509E7,2026-01-28T13:56:32.86Z
ethereum,eth,Ethereum,3033.46,366312531113,2,28853865931,1.206943737963051E8,2026-01-28T13:56:32.866Z
ripple,xrp,XRP,1.93,117521586484,5,2432407258,6.0853233336E10,2026-01-28T13:56:30.246Z
solana,sol,Solana,127.34,72090574319,6,4343364070,5.660685576819917E8,2026-01-28T13:56:29.408Z


In [0]:
df_latest.write \
  .mode("overwrite") \
  .format("delta") \
  .save("abfss://curated@stdeprojectksc.dfs.core.windows.net/crypto_analytics/crypto_latest_snapshot")

In [0]:
df_market.write \
  .mode("overwrite") \
  .format("delta") \
  .save("abfss://curated@stdeprojectksc.dfs.core.windows.net/crypto_analytics/market_summary")

In [0]:
spark.read.format("delta") \
  .load("abfss://curated@stdeprojectksc.dfs.core.windows.net/crypto_analytics/market_summary") \
  .display()

id,symbol,name,current_price,market_cap,market_cap_rank,total_volume,circulating_supply,last_updated_ts
bitcoin,btc,Bitcoin,90187.0,1802957654463,1,49207481311,1.9981509E7,2026-01-28T13:56:32.86Z
ethereum,eth,Ethereum,3033.46,366312531113,2,28853865931,1.206943737963051E8,2026-01-28T13:56:32.866Z
ripple,xrp,XRP,1.93,117521586484,5,2432407258,6.0853233336E10,2026-01-28T13:56:30.246Z
solana,sol,Solana,127.34,72090574319,6,4343364070,5.660685576819917E8,2026-01-28T13:56:29.408Z


In [0]:
from pyspark.sql.functions import col

df_latest_flat = df_latest \
    .withColumn("roi_currency", col("roi.currency")) \
    .withColumn("roi_percentage", col("roi.percentage")) \
    .withColumn("roi_times", col("roi.times")) \
    .drop("roi")


In [0]:
dbutils.fs.ls("abfss://powerbi@stdeprojectksc.dfs.core.windows.net/")

[FileInfo(path='abfss://powerbi@stdeprojectksc.dfs.core.windows.net/crypto_latest/', name='crypto_latest/', size=0, modificationTime=1769786051000)]

In [0]:
df_latest_flat \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("abfss://powerbi@stdeprojectksc.dfs.core.windows.net/crypto_latest")